In [0]:
%run /Shared/insclm_capstone/NB_00_config_loader.py

[SecretScope(name=' kv-insclm-cap-11'), SecretScope(name='kv-insclm')]

[SecretMetadata(key='adls-abfss-base'),
 SecretMetadata(key='adls-account-key'),
 SecretMetadata(key='adls-account-name'),
 SecretMetadata(key='adls-audit-path'),
 SecretMetadata(key='adls-base-url'),
 SecretMetadata(key='adls-bronze-path'),
 SecretMetadata(key='adls-container-name'),
 SecretMetadata(key='adls-gold-path'),
 SecretMetadata(key='adls-raw-path'),
 SecretMetadata(key='adls-rejected-path'),
 SecretMetadata(key='adls-silver-path'),
 SecretMetadata(key='database-workspace-url'),
 SecretMetadata(key='databricks-cluster-id'),
 SecretMetadata(key='databricks-pat'),
 SecretMetadata(key='file-claim-status-updates'),
 SecretMetadata(key='file-claims'),
 SecretMetadata(key='file-customer-master'),
 SecretMetadata(key='file-policy-master'),
 SecretMetadata(key='github-pat'),
 SecretMetadata(key='github-repo-url'),
 SecretMetadata(key='sql-admin-name'),
 SecretMetadata(key='sql-admin-password'),
 SecretMetadata(key='sql-connection-string'),
 SecretMetadata(key='sql-database-name'),
 S

✅ Config loaded from Key Vault successfully.
   ADLS Account  : [REDACTED]
   Container     : [REDACTED]
   ABFSS Base    : [REDACTED]
   RAW path      : [REDACTED][REDACTED]
   BRONZE path   : [REDACTED][REDACTED]
   SILVER path   : [REDACTED][REDACTED]
   GOLD path     : [REDACTED][REDACTED]
   REJECTED path : [REDACTED][REDACTED]
   AUDIT path    : [REDACTED][REDACTED]
   SQL Server    : [REDACTED]
   SQL Database  : [REDACTED]


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("CREATE DATABASE IF NOT EXISTS gold_insclm")
print("✅ gold_insclm database ready")

✅ gold_insclm database ready


In [0]:
print("\n📥 Loading silver tables...")
claims_fact = spark.table("silver_insclm.silver_claims_fact")
policy_dim  = spark.table("silver_insclm.silver_policy_dim")
status_hist = spark.table(
    "silver_insclm.silver_claim_status_history")

print(f"   silver_claims_fact          : {claims_fact.count():,}")
print(f"   silver_policy_dim           : {policy_dim.count():,}")
print(f"   silver_claim_status_history : {status_hist.count():,}")

# Verify policy_type is no longer NULL
print("\npolicy_type distribution:")
claims_fact.groupBy("policy_type") \
    .count().orderBy(F.desc("count")).show()


📥 Loading silver tables...
   silver_claims_fact          : 2,080
   silver_policy_dim           : 1,500
   silver_claim_status_history : 1,600

policy_type distribution:
+-----------+-----+
|policy_type|count|
+-----------+-----+
|     Travel|  475|
|      Motor|  417|
|       Home|  410|
|       Life|  400|
|     Health|  378|
+-----------+-----+



In [0]:
# Latest status per claim
status_window = Window.partitionBy("claim_id") \
    .orderBy(F.desc("status_date"))

latest_status = (status_hist
    .withColumn("rn", F.row_number().over(status_window))
    .filter(F.col("rn") == 1)
    .select(
        "claim_id",
        F.col("new_status").alias("final_status"),
        F.col("status_date").alias("final_status_date"),
        F.col("remarks").alias("final_remarks"),
        "is_terminal_status"))

claims_with_status = claims_fact.join(
    latest_status, "claim_id", "left")

print(f"✅ Latest status joined")
print(f"   Rows: {claims_with_status.count():,}")

✅ Latest status joined
   Rows: 2,080


In [0]:
# Gold Table 1 — gold_claim_summary
print("\n📥 Building gold_claim_summary...")

spark.sql("DROP TABLE IF EXISTS gold_insclm.gold_claim_summary")

gold_claim_summary = (claims_with_status
    .groupBy("policy_type","claim_reason")
    .agg(
        F.count("claim_id")
         .alias("total_claims"),
        F.sum(F.when(F.col("final_status")=="Approved",1)
              .otherwise(0))
         .alias("approved_claims"),
        F.sum(F.when(F.col("final_status")=="Rejected",1)
              .otherwise(0))
         .alias("rejected_claims"),
        F.sum(F.when(F.col("final_status")=="Pending",1)
              .otherwise(0))
         .alias("pending_claims"),
        F.sum(F.when(F.col("final_status")=="Investigation",1)
              .otherwise(0))
         .alias("investigation_claims"),
        F.round(F.sum("claim_amount"),2)
         .alias("total_claim_amount"),
        F.round(F.avg("claim_amount"),2)
         .alias("avg_claim_amount"),
        F.round(F.min("claim_amount"),2)
         .alias("min_claim_amount"),
        F.round(F.max("claim_amount"),2)
         .alias("max_claim_amount"),
    )
    .withColumn("approval_rate_pct",
        F.round(F.col("approved_claims") /
                F.col("total_claims") * 100, 2))
    .withColumn("rejection_rate_pct",
        F.round(F.col("rejected_claims") /
                F.col("total_claims") * 100, 2))
    .withColumn("_gold_loaded_at", F.current_timestamp()))

gold_claim_summary.write \
    .format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("gold_insclm.gold_claim_summary")

g1 = spark.table("gold_insclm.gold_claim_summary").count()
print(f"✅ gold_claim_summary → {g1:,} rows")

print("\nPreview — top 5:")
spark.table("gold_insclm.gold_claim_summary") \
    .orderBy(F.desc("total_claims")) \
    .select("policy_type","claim_reason","total_claims",
            "approved_claims","rejected_claims",
            "approval_rate_pct","rejection_rate_pct") \
    .show(5, truncate=False)


📥 Building gold_claim_summary...
✅ gold_claim_summary → 30 rows

Preview — top 5:
+-----------+-------------------+------------+---------------+---------------+-----------------+------------------+
|policy_type|claim_reason       |total_claims|approved_claims|rejected_claims|approval_rate_pct|rejection_rate_pct|
+-----------+-------------------+------------+---------------+---------------+-----------------+------------------+
|Travel     |Medical Expense    |94          |31             |13             |32.98            |13.83             |
|Home       |Theft              |83          |28             |12             |33.73            |14.46             |
|Home       |Natural Calamity   |82          |32             |12             |39.02            |14.63             |
|Motor      |Travel Cancellation|82          |33             |8              |40.24            |9.76              |
|Travel     |Natural Calamity   |82          |28             |15             |34.15            |18.29    

In [0]:
# Gold Table 2 — gold_policy_history_summary
print("\n📥 Building gold_policy_history_summary...")

spark.sql(
    "DROP TABLE IF EXISTS "
    "gold_insclm.gold_policy_history_summary")

policy_agg = (policy_dim
    .groupBy("policy_id","customer_id")
    .agg(
        F.count("*")
         .alias("total_versions"),
        F.min("effective_date")
         .alias("first_effective_date"),
        F.max("effective_date")
         .alias("last_effective_date"),
        F.max(F.when(F.col("is_current")==True,
                     F.col("policy_status")))
         .alias("current_policy_status"),
        F.max(F.when(F.col("is_current")==True,
                     F.col("coverage_amount")))
         .alias("current_coverage"),
        F.max(F.when(F.col("is_current")==True,
                     F.col("premium_amount")))
         .alias("current_premium"),
        F.max(F.when(F.col("is_current")==True,
                     F.col("policy_type")))
         .alias("current_policy_type"),
        F.min(F.when(F.col("is_current")==False,
                     F.col("coverage_amount")))
         .alias("original_coverage"),
    ))

claims_agg = (claims_fact
    .groupBy("policy_id")
    .agg(
        F.count("claim_id")
         .alias("total_claims_against_policy"),
        F.round(F.sum("claim_amount"),2)
         .alias("total_claim_amount_against_policy"),
    ))

gold_policy_history = (policy_agg
    .join(claims_agg, "policy_id", "left")
    .fillna(0, ["total_claims_against_policy",
                "total_claim_amount_against_policy"])
    .withColumn("coverage_change_pct",
        F.when(
            F.col("original_coverage").isNotNull() &
            (F.col("original_coverage") > 0),
            F.round(
                (F.col("current_coverage") -
                 F.col("original_coverage")) /
                F.col("original_coverage") * 100, 2)
        ).otherwise(F.lit(0.0)))
    .withColumn("_gold_loaded_at", F.current_timestamp()))

gold_policy_history.write \
    .format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("gold_insclm.gold_policy_history_summary")

g2 = spark.table(
    "gold_insclm.gold_policy_history_summary").count()
print(f"✅ gold_policy_history_summary → {g2:,} rows")


📥 Building gold_policy_history_summary...
✅ gold_policy_history_summary → 1,500 rows


In [0]:
# Gold Table 3 — gold_suspicious_claim_summary
print("\n📥 Building gold_suspicious_claim_summary...")

spark.sql(
    "DROP TABLE IF EXISTS "
    "gold_insclm.gold_suspicious_claim_summary")

gold_suspicious = (claims_fact
    .filter(F.col("suspicious_score") >= 1)
    .withColumn("suspicion_level",
        F.when(F.col("suspicious_score")==1, "LOW")
        .when(F.col("suspicious_score")==2, "MEDIUM")
        .otherwise("HIGH"))
    .join(latest_status.select(
              "claim_id","final_status",
              "final_status_date","final_remarks"),
          "claim_id", "left")
    .select(
        "claim_id","policy_id","customer_id",
        "customer_name","state","risk_category",
        "claim_amount","coverage_amount",
        "policy_status_at_claim","document_status",
        "claim_reason","claim_date",
        "customer_claim_count",
        "amount_exceeds_coverage_flag",
        "inactive_policy_flag",
        "incomplete_docs_flag",
        "high_frequency_flag",
        "suspicious_score","suspicion_level",
        "final_status","final_status_date",
        "final_remarks")
    .withColumn("_gold_loaded_at", F.current_timestamp()))

gold_suspicious.write \
    .format("delta").mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("gold_insclm.gold_suspicious_claim_summary")

g3 = spark.table(
    "gold_insclm.gold_suspicious_claim_summary").count()
print(f"✅ gold_suspicious_claim_summary → {g3:,} rows")

print("\nSuspicion level breakdown:")
spark.table("gold_insclm.gold_suspicious_claim_summary") \
    .groupBy("suspicion_level") \
    .count() \
    .orderBy("suspicion_level").show()


📥 Building gold_suspicious_claim_summary...
✅ gold_suspicious_claim_summary → 1,273 rows

Suspicion level breakdown:
+---------------+-----+
|suspicion_level|count|
+---------------+-----+
|           HIGH|   32|
|            LOW|  935|
|         MEDIUM|  306|
+---------------+-----+



In [0]:
print("\n" + "=" * 55)
print("GOLD LAYER SUMMARY")
print("=" * 55)
print(f"gold_claim_summary            : {g1:,} rows")
print(f"gold_policy_history_summary   : {g2:,} rows")
print(f"gold_suspicious_claim_summary : {g3:,} rows")
print("=" * 55)
print("✅ NB_06 complete.")
print("   Next: Run NB_07_time_travel_demo")


GOLD LAYER SUMMARY
gold_claim_summary            : 30 rows
gold_policy_history_summary   : 1,500 rows
gold_suspicious_claim_summary : 1,273 rows
✅ NB_06 complete.
   Next: Run NB_07_time_travel_demo
